In [8]:
import torch
import torch.nn as nn
from sklearn.metrics import confusion_matrix, classification_report
import numpy as np
import torch.nn.functional as F
import pickle

In [11]:
with open('processed/val_data.pkl', 'rb') as f:
    val_data = pickle.load(f)

sequences = torch.cat([torch.from_numpy(s) for s in val_data['sequences']], dim=0)
labels = torch.cat([torch.from_numpy(l) for l in val_data['labels']], dim=0)

print(f"Sequences shape: {sequences.shape}")
print(f"Labels shape: {labels.shape}")

Sequences shape: torch.Size([39164, 4, 8000])
Labels shape: torch.Size([39164])


In [2]:
class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch, kernel = 7, dilation = 1):
        super().__init__()
        padding = (kernel - 1) * dilation // 2
        self.conv = nn.Conv1d(in_ch, out_ch, kernel, dilation=dilation, padding=padding)
        self.bn = nn.BatchNorm1d(out_ch)
        self.dp = nn.Dropout(0.1)

    def forward(self, x):
        return self.dp(F.relu(self.bn(self.conv(x))))

In [3]:
class IsoformClassifier(nn.Module):
    def __init__(self, n_classes = 4):
        super().__init__()
        self.conv1 = ConvBlock(4, 64, kernel = 7, dilation = 1)
        self.conv2 = ConvBlock(64, 64, kernel = 7, dilation = 2)
        self.conv3 = ConvBlock(64, 128, kernel = 7, dilation = 4)
        self.conv4 = ConvBlock(128, 128, kernel = 7, dilation = 8)

        self.classifier = nn.Sequential(
            nn.Linear(128, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, n_classes)
        )
    
    def forward(self, x):
        x = self.conv4(self.conv3(self.conv2(self.conv1(x))))
        x = x.max(dim = -1).values
        return self.classifier(x)

In [15]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = IsoformClassifier(n_classes=4)
model.load_state_dict(torch.load('conv_only_checkpoint.pt', map_location=device))
model.to(device)
model.eval()

batch_size = 32
all_preds = []
all_labels = []

with torch.no_grad():
    for i in range(0, len(sequences), batch_size):
        batch_seq = sequences[i:i+batch_size].to(device)
        batch_labels = labels[i:i+batch_size]
        
        logits = model(batch_seq)
        preds = logits.argmax(dim=1).cpu().numpy()
        
        all_preds.extend(preds)
        all_labels.extend(batch_labels.numpy())
        
        if (i // batch_size) % 100 == 0:
            print(f"Processed {i} / {len(sequences)}")

all_preds = np.array(all_preds)
all_labels = np.array(all_labels)

from sklearn.metrics import confusion_matrix, classification_report
cm = confusion_matrix(all_labels, all_preds)
print(cm)
print(classification_report(all_labels, all_preds, 
                        target_names=['real_high', 'real_mid', 'uncertain', 'artifact']))

Processed 0 / 39164
Processed 3200 / 39164
Processed 6400 / 39164
Processed 9600 / 39164
Processed 12800 / 39164
Processed 16000 / 39164
Processed 19200 / 39164
Processed 22400 / 39164
Processed 25600 / 39164
Processed 28800 / 39164
Processed 32000 / 39164
Processed 35200 / 39164
Processed 38400 / 39164
[[10465    64  5650    15]
 [ 1282   685  1580    26]
 [ 1448     6 12546    45]
 [  521    14  3724  1093]]
              precision    recall  f1-score   support

   real_high       0.76      0.65      0.70     16194
    real_mid       0.89      0.19      0.32      3573
   uncertain       0.53      0.89      0.67     14045
    artifact       0.93      0.20      0.33      5352

    accuracy                           0.63     39164
   macro avg       0.78      0.48      0.50     39164
weighted avg       0.71      0.63      0.60     39164



In [16]:
unique, counts = np.unique(all_labels, return_counts=True)
for cls, cnt in zip(unique, counts):
    class_names = ['real_high', 'real_mid', 'uncertain', 'artifact']
    print(f"{class_names[cls]}: {cnt} ({100*cnt/len(all_labels):.1f}%)")

real_high: 16194 (41.3%)
real_mid: 3573 (9.1%)
uncertain: 14045 (35.9%)
artifact: 5352 (13.7%)
